## Part 1: The Tokenizers

### Pre-experiment Predictions

**Which language do you expect to require the most tokens?**
 - Turkish to require the most tokens.
   - Turkish and English both use Latin-based alphabets, but Turkish has _"more productive word morphology"_.
   - Individual Chinese characters are often words on their own.

**How do you expect the larger BPE vocabulary to change the representation of each language?**
 - English — full words or even common short phrases.
 - Turkish — I think it will be the same as English.
 - Chinese — common 2 or more character words and even idioms

**What kinds of units do you expect BPE to learn in English, Turkish, and Chinese?**
 - English — frequently used words (e.g. the, and, a, etc...) and common prefixes and suffixes (e.g. -ing, -er, re-, etc...)
 - Turkish — I think it will be the same as English.
 - Chinese — frequently used characters (e.g. 的，了，个) and common short words.

**Which tokenizer do you expect to work best for language modeling?**
 - I'm guessing the small-sized BPE tokenizer might work best for language modeling.
   - Character-level tokenization will be too slow and inefficient, and making it hard for the model to manage the context.
   - If the BPE vocabulary is too large, there will be a lot more parameters for the model.
   - If the BPE vocabulary is small and compressed at the sweet spot, it is more efficient for the model.


### Pre-experiment Observation

I noticed that the Chinese corpus contains a mixture of traditional and simplified Chinese characters (e.g. 大學 and 大学). Both have the same meaning but would impact the frequencies since they are considered different tokens. This affects evaluations further down the pipeline.

As such, I have decided to do a normalization by converting all the Chinese characters to simplified Chinese characters for the entire dataset.

To run the normalization, run the following command: `uv run normalize-chinese.py`.


In [ ]:
#!uv run normalize-chinese.py

#### Character-level Tokenizer

In [ ]:
unique_chars = set()

with open('./data/tokenizer/balanced.txt', 'r', encoding='utf-8') as f:
    for line in f:
        unique_chars.update(line.strip())

unique_chars = sorted(unique_chars)

print(len(unique_chars))
print(unique_chars[:10])

In [ ]:
UNK_TOKEN = '<UNK>'
PAD_TOKEN = '<PAD>'
BOS_TOKEN = '<BOS>'
EOS_TOKEN = '<EOS>'
SPECIAL_TOKENS = [UNK_TOKEN, PAD_TOKEN, BOS_TOKEN, EOS_TOKEN]
UNK_ID, PAD_ID, BOS_ID, EOS_ID = range(len(SPECIAL_TOKENS))

vocab = SPECIAL_TOKENS + unique_chars

print(len(vocab))
print(vocab[:10])

In [ ]:
charToId = {char: id for id, char in enumerate(vocab)}
idToChar = {id: char for id, char in enumerate(vocab)}

print(list(charToId.items())[:10])
print(list(idToChar.items())[:10])

In [ ]:
def encode(text):
    ids = []
    for char in text:
        ids.append(charToId.get(char, UNK_ID))
    return ids


def decode(ids):
    chars = []
    for id in ids:
        chars.append(idToChar.get(id, UNK_TOKEN))
    return ''.join(chars)


print(encode('hello'), decode(encode('hello')))
print(encode("✨"), UNK_ID)

#### BPE Tokenizer

In [ ]:
# %%time
import os

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer


def train_bpe(vocab_size, input_file, output_file):
    if os.path.exists(output_file):
        print(f"Tokenizer already exists at {output_file}. Skipping training.")
        return Tokenizer.from_file(output_file)

    tokenizer = Tokenizer(BPE(unk_token=UNK_TOKEN))
    trainer = BpeTrainer(special_tokens=SPECIAL_TOKENS, vocab_size=vocab_size)
    tokenizer.pre_tokenizer = Whitespace()
    tokenizer.train([input_file], trainer)
    tokenizer.save(output_file)
    return tokenizer


tokenizer_small = train_bpe(
    10000, './data/tokenizer/balanced.txt', 'data/tokenizer/bpe-tokenizer-small.json')
tokenizer_big = train_bpe(
    30000, './data/tokenizer/balanced.txt', 'data/tokenizer/bpe-tokenizer-big.json')

**Reason for BPE Vocabulary Sizes**

#TODO